# Pretrained ESM-2 forward-pass profile

This notebook profiles the exact pretrained teacher used by `encode_proteins_from_fasta.py`: `esm2_t33_650M_UR50D`, layer-33 representations, evaluation mode, FP32, and no contact prediction. It answers three questions:

1. What are the forward latency, throughput, and peak CUDA memory for the real production shape?
2. Which model components and transformer layers consume the most CUDA time?
3. Which PyTorch operators explain that component-level cost?

Tokenization, host-to-device transfer, mean pooling, and file I/O are deliberately outside the timed region. CUDA events are synchronized before reading results. The first executions are warm-ups, and reported latencies are medians over repeated forwards.

In [ ]:
from __future__ import annotations

import os
import platform
from collections import defaultdict
from pathlib import Path
from statistics import median

import esm
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch.profiler import ProfilerActivity, profile

MODEL_FACTORY = "esm2_t33_650M_UR50D"
REPRESENTATION_LAYER = 33
FASTA_PATH = Path(
    os.environ.get(
        "ESM_PROFILE_FASTA",
        "protein_data/filtered_msa.fasta",
    )
)
TRUNCATE_SEQUENCE_LENGTH = 1307
TOKEN_BUDGET = 4096
GPU_INDEX = int(os.environ.get("ESM_PROFILE_GPU", "0"))
WARMUP_STEPS = int(os.environ.get("ESM_PROFILE_WARMUPS", "3"))
TIMED_STEPS = int(os.environ.get("ESM_PROFILE_STEPS", "10"))
MODULE_PROFILE_STEPS = int(os.environ.get("ESM_PROFILE_MODULE_STEPS", "8"))

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires CUDA for synchronized component timing.")
if GPU_INDEX >= torch.cuda.device_count():
    raise ValueError(
        f"ESM_PROFILE_GPU={GPU_INDEX}, but only {torch.cuda.device_count()} GPUs are visible."
    )

DEVICE = torch.device(f"cuda:{GPU_INDEX}")
torch.cuda.set_device(DEVICE)
torch.manual_seed(0)
pd.options.display.float_format = "{:,.3f}".format

environment = pd.DataFrame(
    [
        ("Model", MODEL_FACTORY),
        ("Representation layer", REPRESENTATION_LAYER),
        ("Python", platform.python_version()),
        ("PyTorch", torch.__version__),
        ("fair-esm", getattr(esm, "__version__", "2.0.0")),
        ("GPU index", GPU_INDEX),
        ("GPU", torch.cuda.get_device_name(DEVICE)),
        ("CUDA capability", ".".join(map(str, torch.cuda.get_device_capability(DEVICE)))),
        ("Precision", "FP32"),
        ("Mode", "eval + torch.no_grad"),
    ],
    columns=["Setting", "Value"],
)
environment

## Workload discovery

The production FASTA determines sequence length, while the encoder's token budget determines batch size. Reading the FASTA here prevents a synthetic benchmark from silently drifting away from the real workload.

In [ ]:
def summarize_fasta(path: Path, sample_count: int = 3) -> tuple[list[int], list[tuple[str, str]]]:
    """Return all sequence lengths and the first few complete FASTA records."""
    if not path.exists():
        raise FileNotFoundError(
            f"FASTA not found at {path}. Set ESM_PROFILE_FASTA before starting the kernel."
        )

    lengths: list[int] = []
    samples: list[tuple[str, str]] = []
    current_label: str | None = None
    current_parts: list[str] = []

    def finish_record() -> None:
        if current_label is None:
            return
        sequence = "".join(current_parts).upper().replace(".", "-")
        lengths.append(len(sequence))
        if len(samples) < sample_count:
            samples.append((current_label, sequence))

    with path.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith(">"):
                finish_record()
                current_label = line[1:]
                current_parts = []
            else:
                current_parts.append(line)
    finish_record()
    return lengths, samples


sequence_lengths, sample_records = summarize_fasta(FASTA_PATH)
length_series = pd.Series(sequence_lengths, name="residues")
length_summary = pd.DataFrame(
    {
        "sequences": [len(length_series)],
        "min": [length_series.min()],
        "p50": [length_series.quantile(0.50)],
        "p90": [length_series.quantile(0.90)],
        "p99": [length_series.quantile(0.99)],
        "max": [length_series.max()],
        "mean": [length_series.mean()],
    }
)
representative_residues = int(length_series.median())
tokens_per_sequence = min(representative_residues, TRUNCATE_SEQUENCE_LENGTH) + 2
production_batch_size = max(TOKEN_BUDGET // tokens_per_sequence, 1)

display(length_summary)
display(
    Markdown(
        f"**Production shape:** `{production_batch_size} x {tokens_per_sequence}` tokens "
        f"from a `{TOKEN_BUDGET}`-token budget (BOS and EOS included)."
    )
)

## Load the pretrained teacher

The first run downloads the official weights into PyTorch's checkpoint cache. The production batch uses real sequences from the configured FASTA and fair-esm's native batch converter.

In [ ]:
model_loader = getattr(esm.pretrained, MODEL_FACTORY)
model, alphabet = model_loader()
model = model.eval().to(DEVICE)

if len(sample_records) < production_batch_size:
    raise RuntimeError("The FASTA does not contain enough records for one production batch.")

batch_converter = alphabet.get_batch_converter(TRUNCATE_SEQUENCE_LENGTH)
selected_records = sample_records[:production_batch_size]
_, _, production_tokens_cpu = batch_converter(selected_records)
production_tokens = production_tokens_cpu.to(DEVICE)
single_tokens = production_tokens[:1]

parameter_rows = [
    ("Token embedding", sum(p.numel() for p in model.embed_tokens.parameters())),
    ("Transformer blocks", sum(p.numel() for p in model.layers.parameters())),
    ("Final layer norm", sum(p.numel() for p in model.emb_layer_norm_after.parameters())),
    ("Language-model head", sum(p.numel() for p in model.lm_head.parameters())),
    ("Contact head", sum(p.numel() for p in model.contact_head.parameters())),
]
parameter_table = pd.DataFrame(parameter_rows, columns=["Component", "Parameters"])
parameter_table["Million parameters"] = parameter_table["Parameters"] / 1e6
parameter_table["Share of model (%)"] = (
    100 * parameter_table["Parameters"] / sum(p.numel() for p in model.parameters())
)

display(
    Markdown(
        f"Loaded **{sum(p.numel() for p in model.parameters()) / 1e6:,.1f}M parameters**; "
        f"timed tensor shape is `{tuple(production_tokens.shape)}`."
    )
)
parameter_table

## End-to-end forward latency and memory

Each sample below is one complete `model(tokens, repr_layers=[33], return_contacts=False)` call. The model still computes its 33-way language-model logits internally; contact prediction is disabled.

In [ ]:
def measure_forward(tokens: torch.Tensor, warmups: int, steps: int) -> dict[str, float]:
    """Measure synchronized CUDA-event latency and peak allocated memory."""
    with torch.no_grad():
        for _ in range(warmups):
            output = model(
                tokens,
                repr_layers=[REPRESENTATION_LAYER],
                return_contacts=False,
            )
            del output
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)

    latencies_ms: list[float] = []
    with torch.no_grad():
        for _ in range(steps):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            output = model(
                tokens,
                repr_layers=[REPRESENTATION_LAYER],
                return_contacts=False,
            )
            end.record()
            end.synchronize()
            latencies_ms.append(start.elapsed_time(end))
            del output

    latency_series = pd.Series(latencies_ms)
    latency_median = float(latency_series.median())
    return {
        "Batch size": int(tokens.shape[0]),
        "Tokens / sequence": int(tokens.shape[1]),
        "Median latency (ms)": latency_median,
        "Mean latency (ms)": float(latency_series.mean()),
        "Std. dev. (ms)": float(latency_series.std(ddof=1)),
        "P10 latency (ms)": float(latency_series.quantile(0.10)),
        "P90 latency (ms)": float(latency_series.quantile(0.90)),
        "Sequences / second": 1000.0 * tokens.shape[0] / latency_median,
        "Tokens / second": 1000.0 * tokens.numel() / latency_median,
        "Peak allocated (GiB)": torch.cuda.max_memory_allocated(DEVICE) / 2**30,
    }


latency_table = pd.DataFrame(
    [
        measure_forward(single_tokens, WARMUP_STEPS, TIMED_STEPS),
        measure_forward(production_tokens, WARMUP_STEPS, TIMED_STEPS),
    ]
)
latency_table

## Hierarchical module timing

CUDA-event hooks measure inclusive time for the whole block, attention, each projection, both FFN projections, and both layer norms. Parent-minus-child subtraction provides the remaining attention core and elementwise/block overhead. Hooks add a small amount of launch overhead, so use the previous table for absolute latency and this section for component shares.

In [ ]:
def timed_module_names(num_layers: int) -> list[str]:
    """List modules needed for the hierarchical timing decomposition."""
    names = ["embed_tokens", "emb_layer_norm_after", "lm_head"]
    for layer_index in range(num_layers):
        prefix = f"layers.{layer_index}"
        names.extend(
            [
                prefix,
                f"{prefix}.self_attn",
                f"{prefix}.self_attn.q_proj",
                f"{prefix}.self_attn.k_proj",
                f"{prefix}.self_attn.v_proj",
                f"{prefix}.self_attn.out_proj",
                f"{prefix}.self_attn_layer_norm",
                f"{prefix}.final_layer_norm",
                f"{prefix}.fc1",
                f"{prefix}.fc2",
            ]
        )
    return names


def collect_module_timings(tokens: torch.Tensor, steps: int) -> list[dict[str, float]]:
    """Collect one dictionary of synchronized module durations per forward."""
    modules = dict(model.named_modules())
    names = timed_module_names(model.num_layers)
    missing = sorted(set(names) - set(modules))
    if missing:
        raise KeyError(f"Expected fair-esm modules were not found: {missing}")

    records: list[tuple[str, torch.cuda.Event, torch.cuda.Event]] = []
    active_starts: dict[str, list[torch.cuda.Event]] = defaultdict(list)
    handles: list[torch.utils.hooks.RemovableHandle] = []

    def make_pre_hook(name: str):
        def pre_hook(_module, _inputs):
            event = torch.cuda.Event(enable_timing=True)
            event.record()
            active_starts[name].append(event)
        return pre_hook

    def make_post_hook(name: str):
        def post_hook(_module, _inputs, _output):
            event = torch.cuda.Event(enable_timing=True)
            event.record()
            records.append((name, active_starts[name].pop(), event))
        return post_hook

    for name in names:
        handles.append(modules[name].register_forward_pre_hook(make_pre_hook(name)))
        handles.append(modules[name].register_forward_hook(make_post_hook(name)))

    runs: list[dict[str, float]] = []
    try:
        with torch.no_grad():
            for _ in range(steps):
                records.clear()
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                start.record()
                output = model(
                    tokens,
                    repr_layers=[REPRESENTATION_LAYER],
                    return_contacts=False,
                )
                end.record()
                end.synchronize()

                durations: dict[str, float] = defaultdict(float)
                for name, module_start, module_end in records:
                    durations[name] += module_start.elapsed_time(module_end)
                durations["forward"] = start.elapsed_time(end)
                runs.append(dict(durations))
                del output
    finally:
        for handle in handles:
            handle.remove()
    return runs


module_runs = collect_module_timings(production_tokens, MODULE_PROFILE_STEPS)
instrumented_latencies = pd.Series([run["forward"] for run in module_runs])
display(
    Markdown(
        f"Instrumented forward median: **{instrumented_latencies.median():.2f} ms** "
        f"(range {instrumented_latencies.min():.2f}-{instrumented_latencies.max():.2f} ms)."
    )
)

In [ ]:
layer_names = [f"layers.{index}" for index in range(model.num_layers)]
attention_names = [f"{name}.self_attn" for name in layer_names]
fc1_names = [f"{name}.fc1" for name in layer_names]
fc2_names = [f"{name}.fc2" for name in layer_names]
norm_names = [
    module_name
    for layer_name in layer_names
    for module_name in (
        f"{layer_name}.self_attn_layer_norm",
        f"{layer_name}.final_layer_norm",
    )
]
projection_groups = {
    "Query projections": [f"{name}.self_attn.q_proj" for name in layer_names],
    "Key projections": [f"{name}.self_attn.k_proj" for name in layer_names],
    "Value projections": [f"{name}.self_attn.v_proj" for name in layer_names],
    "Output projections": [f"{name}.self_attn.out_proj" for name in layer_names],
}

def run_sum(run: dict[str, float], names: list[str]) -> float:
    return sum(run[name] for name in names)

def median_run_sum(names: list[str]) -> float:
    return median(run_sum(run, names) for run in module_runs)

instrumented_forward_ms = float(instrumented_latencies.median())
top_level_values = {
    "Token embedding module": median(run["embed_tokens"] for run in module_runs),
    "Transformer blocks (33)": median_run_sum(layer_names),
    "Final layer norm": median(run["emb_layer_norm_after"] for run in module_runs),
    "Language-model head": median(run["lm_head"] for run in module_runs),
}
top_level_values["Top-level tensor/framework work"] = max(
    instrumented_forward_ms - sum(top_level_values.values()), 0.0
)

block_ms = top_level_values["Transformer blocks (33)"]
attention_ms = median_run_sum(attention_names)
fc1_ms = median_run_sum(fc1_names)
fc2_ms = median_run_sum(fc2_names)
norm_ms = median_run_sum(norm_names)
block_other_ms = max(block_ms - attention_ms - fc1_ms - fc2_ms - norm_ms, 0.0)

top_level_table = pd.DataFrame(
    [(name, value) for name, value in top_level_values.items()],
    columns=["Component", "Median CUDA time (ms)"],
)
top_level_table["Forward share (%)"] = (
    100 * top_level_table["Median CUDA time (ms)"] / instrumented_forward_ms
)

block_table = pd.DataFrame(
    [
        ("Self-attention (inclusive)", attention_ms),
        ("FFN input projection (fc1)", fc1_ms),
        ("FFN output projection (fc2)", fc2_ms),
        ("Two layer norms", norm_ms),
        ("GELU, residuals, dropout, and block overhead", block_other_ms),
    ],
    columns=["Transformer subcomponent", "Median CUDA time (ms)"],
)
block_table["Block share (%)"] = 100 * block_table["Median CUDA time (ms)"] / block_ms
block_table["Forward share (%)"] = (
    100 * block_table["Median CUDA time (ms)"] / instrumented_forward_ms
)

projection_values = {
    group_name: median_run_sum(group_names)
    for group_name, group_names in projection_groups.items()
}
projection_values["Attention core (RoPE + QK^T + softmax + AV)"] = max(
    attention_ms - sum(projection_values.values()), 0.0
)
attention_table = pd.DataFrame(
    [(name, value) for name, value in projection_values.items()],
    columns=["Attention subcomponent", "Median CUDA time (ms)"],
)
attention_table["Attention share (%)"] = (
    100 * attention_table["Median CUDA time (ms)"] / attention_ms
)
attention_table["Forward share (%)"] = (
    100 * attention_table["Median CUDA time (ms)"] / instrumented_forward_ms
)

display(Markdown("### Whole forward"))
display(top_level_table.sort_values("Median CUDA time (ms)", ascending=False))
display(Markdown("### Inside all 33 transformer blocks"))
display(block_table.sort_values("Median CUDA time (ms)", ascending=False))
display(Markdown("### Inside self-attention"))
display(attention_table.sort_values("Median CUDA time (ms)", ascending=False))

In [ ]:
layer_rows = []
for layer_index, layer_name in enumerate(layer_names, start=1):
    layer_rows.append(
        {
            "Layer": layer_index,
            "Whole block (ms)": median(run[layer_name] for run in module_runs),
            "Attention (ms)": median(
                run[f"{layer_name}.self_attn"] for run in module_runs
            ),
            "fc1 (ms)": median(run[f"{layer_name}.fc1"] for run in module_runs),
            "fc2 (ms)": median(run[f"{layer_name}.fc2"] for run in module_runs),
            "Norms (ms)": median(
                run[f"{layer_name}.self_attn_layer_norm"]
                + run[f"{layer_name}.final_layer_norm"]
                for run in module_runs
            ),
        }
    )
layer_table = pd.DataFrame(layer_rows)
layer_table["Forward share (%)"] = 100 * layer_table["Whole block (ms)"] / instrumented_forward_ms

display(Markdown("### Slowest individual transformer blocks"))
display(layer_table.nlargest(10, "Whole block (ms)"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
top_level_table.sort_values("Median CUDA time (ms)").plot.barh(
    x="Component", y="Median CUDA time (ms)", legend=False, ax=axes[0], color="#2878B5"
)
axes[0].set_title("Whole forward")
axes[0].set_xlabel("Median CUDA time (ms)")
axes[0].set_ylabel("")

axes[1].plot(layer_table["Layer"], layer_table["Whole block (ms)"], label="Whole block")
axes[1].plot(layer_table["Layer"], layer_table["Attention (ms)"], label="Attention")
axes[1].plot(
    layer_table["Layer"],
    layer_table["fc1 (ms)"] + layer_table["fc2 (ms)"],
    label="fc1 + fc2",
)
axes[1].set_title("Per-layer CUDA time")
axes[1].set_xlabel("Transformer layer")
axes[1].set_ylabel("Median CUDA time (ms)")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Operator-level CUDA profile

One additional production-shape forward is captured with `torch.profiler`. Only `aten::` rows are retained so CUDA kernel activity is not double-counted alongside its parent PyTorch operator.

In [ ]:
with torch.no_grad(), profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=False,
    profile_memory=True,
) as torch_profile:
    profiler_output = model(
        production_tokens,
        repr_layers=[REPRESENTATION_LAYER],
        return_contacts=False,
    )
torch.cuda.synchronize(DEVICE)
del profiler_output

operator_rows = []
for event in torch_profile.key_averages():
    self_cuda_us = float(event.self_device_time_total)
    if event.key.startswith("aten::") and self_cuda_us > 0:
        operator_rows.append(
            {
                "Operator": event.key,
                "Calls": int(event.count),
                "Self CUDA time (ms)": self_cuda_us / 1000.0,
                "Mean self CUDA (us/call)": self_cuda_us / event.count,
                "Self CUDA memory (MiB)": event.self_device_memory_usage / 2**20,
            }
        )

operator_table = pd.DataFrame(operator_rows).sort_values(
    "Self CUDA time (ms)", ascending=False
)
aten_cuda_total_ms = operator_table["Self CUDA time (ms)"].sum()
operator_table["ATen CUDA share (%)"] = (
    100 * operator_table["Self CUDA time (ms)"] / aten_cuda_total_ms
)
operator_table.head(20)

## Generated report

The report below is assembled from the measured tables, so it remains internally consistent after changing the GPU, sequence data, or repeat counts.

In [ ]:
production_latency = latency_table.loc[latency_table["Batch size"] == production_batch_size].iloc[0]
single_latency = latency_table.loc[latency_table["Batch size"] == 1].iloc[0]
slowest_layer = layer_table.loc[layer_table["Whole block (ms)"].idxmax()]
fastest_layer = layer_table.loc[layer_table["Whole block (ms)"].idxmin()]
top_operators = operator_table.head(5).reset_index(drop=True)

report = f"""
### Executive conclusion

For the repository's real **{production_batch_size} x {tokens_per_sequence} FP32** workload on an
**{torch.cuda.get_device_name(DEVICE)}**, a complete pretrained ESM-2 650M forward takes
**{production_latency['Median latency (ms)']:.2f} ms median**
(P10-P90: {production_latency['P10 latency (ms)']:.2f}-{production_latency['P90 latency (ms)']:.2f} ms),
or **{production_latency['Sequences / second']:.2f} sequences/s**. Peak allocated CUDA memory is
**{production_latency['Peak allocated (GiB)']:.2f} GiB**, including the model.

The answer to "what takes the most time?" is the **33 transformer blocks**:
**{top_level_values['Transformer blocks (33)']:.2f} ms** or
**{100 * top_level_values['Transformer blocks (33)'] / instrumented_forward_ms:.1f}%** of the instrumented forward.
Within them, **self-attention is the largest component** at **{attention_ms:.2f} ms**
({100 * attention_ms / instrumented_forward_ms:.1f}% of the whole forward), followed by the two FFN matrix
projections at **{fc1_ms + fc2_ms:.2f} ms combined**
({100 * (fc1_ms + fc2_ms) / instrumented_forward_ms:.1f}%).

### Component accounting

- Attention's non-projection core (RoPE, QK^T, softmax, and probability-value multiplication) costs
  **{projection_values['Attention core (RoPE + QK^T + softmax + AV)']:.2f} ms**,
  **{100 * projection_values['Attention core (RoPE + QK^T + softmax + AV)'] / attention_ms:.1f}%** of attention.
- The four Q/K/V/output projection families cost **{sum(value for key, value in projection_values.items() if key != 'Attention core (RoPE + QK^T + softmax + AV)'):.2f} ms** combined.
- The FFN input and output projections are nearly symmetric: **{fc1_ms:.2f} ms** for `fc1` and
  **{fc2_ms:.2f} ms** for `fc2` across all layers.
- Both layer norms across all blocks cost only **{norm_ms:.2f} ms**
  ({100 * norm_ms / instrumented_forward_ms:.1f}% of the forward).
- The final language-model head costs **{top_level_values['Language-model head']:.2f} ms**
  ({100 * top_level_values['Language-model head'] / instrumented_forward_ms:.2f}%). It is computed by fair-esm even
  though the encoder only consumes `representations[33]`; skipping it would be correct for embedding-only inference,
  but it is not the main bottleneck for ESM's 33-token vocabulary.
- Individual blocks are broadly uniform. Layer {int(slowest_layer['Layer'])} is slowest at
  **{slowest_layer['Whole block (ms)']:.2f} ms**, versus **{fastest_layer['Whole block (ms)']:.2f} ms** for the fastest
  layer (layer {int(fastest_layer['Layer'])}).

### Operator evidence

The leading ATen CUDA operators are:

1. `{top_operators.loc[0, 'Operator']}`: **{top_operators.loc[0, 'Self CUDA time (ms)']:.2f} ms** ({top_operators.loc[0, 'ATen CUDA share (%)']:.1f}%)
2. `{top_operators.loc[1, 'Operator']}`: **{top_operators.loc[1, 'Self CUDA time (ms)']:.2f} ms** ({top_operators.loc[1, 'ATen CUDA share (%)']:.1f}%)
3. `{top_operators.loc[2, 'Operator']}`: **{top_operators.loc[2, 'Self CUDA time (ms)']:.2f} ms** ({top_operators.loc[2, 'ATen CUDA share (%)']:.1f}%)
4. `{top_operators.loc[3, 'Operator']}`: **{top_operators.loc[3, 'Self CUDA time (ms)']:.2f} ms** ({top_operators.loc[3, 'ATen CUDA share (%)']:.1f}%)
5. `{top_operators.loc[4, 'Operator']}`: **{top_operators.loc[4, 'Self CUDA time (ms)']:.2f} ms** ({top_operators.loc[4, 'ATen CUDA share (%)']:.1f}%)

`aten::addmm` is the linear-layer work; `aten::bmm` and `aten::_softmax` are the quadratic attention path.
Together, the module and operator views agree that attention is the first optimization target at sequence length
{representative_residues}, while FFN GEMMs are the second.

### Batch-size reference

A single sequence takes **{single_latency['Median latency (ms)']:.2f} ms** median. Packing three same-length sequences
raises latency to **{production_latency['Median latency (ms)']:.2f} ms** but improves throughput from
**{single_latency['Sequences / second']:.2f}** to **{production_latency['Sequences / second']:.2f} sequences/s**.

### Scope and interpretation

These are inference measurements for fair-esm {getattr(esm, '__version__', '2.0.0')} and PyTorch {torch.__version__}.
They exclude tokenization, host-to-device copies, the caller's mean pooling, CPU copies, and `torch.save`.
CUDA-event module hooks slightly perturb absolute latency, so the uninstrumented latency table is authoritative for
end-to-end numbers and the hook table is used for the component shares. GPU clocks and competing server workloads can
move absolute numbers; rerun the notebook on an idle target GPU before using them as deployment guarantees.
"""

display(Markdown(report))